In [329]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import numpy

In [330]:
#read in the new master file with the developments called master_devs
master_devs = gpd.read_file("data/master_developments.geojson").to_crs(epsg = 4326)
master_devs = master_devs[~master_devs.geometry.isna()]
master_devs

,A_NUMBER,A_TYPE,A_DATE,A_STATUS,A_STATUS_D,A_PROJECT_,A_DESCRIPT,A_USER_ID,A_CASE_PLA,StatCode,...,EditDate,Editor,sf_detached,sf_attached,duplex_triplex,multifamily,condo,unknown,data_source,geometry
0,D1500106,PL_SSP_SM2,2015-05-05,APP,2015-07-10,Garrett Ridge Multi-Family Community,Revise entry and sidewalk locations for end un...,DACIARE,NIARO,APP,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,0.0,0,0,None,POINT (-78.97194 35.95188)
1,D2000291,PL_MINSP,2020-12-02,APP,2021-09-03,Umstead Grove Conservation Subdivision,"50 Single - family lots, 1 stormwater pond, ad...",JESSICADO,COURTNEYMC,APP,...,2025-09-20,gisproc_sys,50.0,0.0,0.0,0.0,0,0,DRHM,POINT (-78.94561 36.07501)
2,D1900396,PL_SSP_SM2,2019-11-12,APP,2020-03-31,Ross Road Apartments,"Renovations to parking, paving, dumpster enclo...",JOHNRA,JOHNRA,APP,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,0.0,0,0,None,POINT (-78.8457 35.99124)
3,D2200173,PL_MINSP,2022-05-25,APP,2023-02-01,Pineview Glen Town homes - Mass Grading,Mass grading only site plan for residential to...,JUSTINH,KEAGANSA,APP,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,0.0,0,0,None,POINT (-78.81051 35.92776)
4,D1800099,PL_MINSP,2018-04-03,APP,2018-08-09,Snikroc,Townhome Residential Development,KARENPA,TREYFI,APP,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,0.0,0,0,None,POINT (-78.79031 35.9733)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584,D2500199,PL_MINSP,2025-08-11,UN_RE,2025-08-11,Howard's Place,Mixed use multi-family development with access...,HARMONHE,TREYFI,UN_RE,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,0.0,0,0,None,POINT (-78.76658 35.97569)
585,D2500202,PL_MINSP,2025-08-14,UN_RE,2025-08-14,443 Infinity Rd,28 unit townhouse community.,ERIKA,TREYFI,UN_RE,...,2025-09-20,gisproc_sys,0.0,28.0,0.0,0.0,0,0,DRHM,POINT (-78.90088 36.079)
586,D2500218,PL_MINSP,2025-09-02,UN_RE,2025-09-02,Green Hill,Construction of 440 multi-family units with as...,MICHAELEVE,JALISAHA,UN_RE,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,440.0,0,0,DRHM,POINT (-78.94701 36.02156)
587,D2500219,PL_MINSP,2025-09-04,UN_RE,2025-09-04,Pearson Flats,Multi-family development to be multiple apartm...,JALISAHA,JALISAHA,UN_RE,...,2025-09-20,gisproc_sys,0.0,0.0,0.0,0.0,0,0,None,POINT (-78.90531 35.95507)


In [331]:
#read in the regions file
durham_regions = gpd.read_file("data/durham_regions.geojson").to_crs(epsg = 4326)
durham_regions

,OBJECTID,region,Shape_Length,Shape_Area,geometry
0,1,Central,98527.060982,3.329115e+08,"MULTIPOLYGON (((-78.89217 36.0348, -78.89154 3..."
1,2,East,188381.789318,1.877839e+09,"MULTIPOLYGON (((-78.7402 36.02356, -78.74019 3..."
2,3,North,271156.004076,3.721947e+09,"MULTIPOLYGON (((-78.80489 36.08853, -78.80558 ..."
3,4,Southeast,168558.327819,1.137551e+09,"MULTIPOLYGON (((-78.86421 35.95016, -78.86332 ..."
4,5,Southwest,177993.213035,1.232725e+09,"MULTIPOLYGON (((-78.90379 35.94973, -78.9038 3..."


In [332]:
#read in the SGRs by region and housing type
sgr_by_region = pd.read_csv("data/sgr_tables_htype_reg.csv").dropna()

#keep only sgr, hytpe, and region
sgr_by_region = sgr_by_region[["housing_type", "region", "sgr_dps_avg_k12"]]

#keep only the 5 types that we have numbers on
kept_htype = ["condo", "du_tri", "mf_apt", "sf_attach", "sf_detach"]
sgr_by_region = sgr_by_region[sgr_by_region['housing_type'].isin(kept_htype)].reset_index(drop=True)

#rename columns to the same ones as the master file for consistency
sgr_by_region['housing_type'] = sgr_by_region['housing_type'].replace('du_tri', 'duplex_triplex')
sgr_by_region['housing_type'] = sgr_by_region['housing_type'].replace('mf_apt', 'multifamily')
sgr_by_region['housing_type'] = sgr_by_region['housing_type'].replace('sf_attach', 'sf_attached')
sgr_by_region['housing_type'] = sgr_by_region['housing_type'].replace('sf_detach', 'sf_detached')

sgr_by_region

,housing_type,region,sgr_dps_avg_k12
0,condo,Central,0.023707
1,duplex_triplex,Central,0.218235
2,multifamily,Central,0.164646
3,sf_attached,Central,0.032995
4,sf_detached,Central,0.262198
5,condo,East,0.573991
6,duplex_triplex,East,0.407080
7,multifamily,East,0.215372
8,sf_attached,East,0.069745
9,sf_detached,East,0.218413


In [333]:
#add in the regions column based on the geometry of each place, drop the ones which don't fall into a region
for i,geometry in enumerate(durham_regions['geometry']):
    in_geometry = geometry.contains(master_devs['geometry'])
    region = durham_regions.loc[i, 'region']

    master_devs.loc[in_geometry, 'region'] = region

master_devs = master_devs[master_devs['region'].notna()].reset_index(drop=True)

In [334]:
#convert to matrix
sgr_matrix = sgr_by_region.pivot(index='region', columns='housing_type', values='sgr_dps_avg_k12')

#sgr function
def compute_stu_gen(row):
    region = row['region']
    # Multiply unit counts by SGR for this region
    return sum(row[h_type] * sgr_matrix.loc[region, h_type] for h_type in ['sf_detached', 'sf_attached', 'condo', 'multifamily', 'duplex_triplex'])

master_devs['stu_gen_100'] = master_devs.apply(compute_stu_gen, axis=1)
master_devs['stu_gen_50'] = master_devs['stu_gen_100']*0.5

In [335]:
#filter out the years
master_devs['A_STATUS_D'] = pd.to_datetime(master_devs['A_STATUS_D'], errors='coerce')
master_devs_2020 = master_devs[master_devs['A_STATUS_D'].dt.year >= 2020].reset_index(drop=True)
master_devs_2023 = master_devs[master_devs['A_STATUS_D'].dt.year >= 2023].reset_index(drop=True)

print(master_devs_2023['stu_gen_100'].sum())

4568.269356277


In [336]:
#master_devs_2020.to_file('data/master_devs_with_stu_gen_2020.geojson', driver = "geojson")
#master_devs_2023.to_file('data/master_devs_with_stu_gen_2023.geojson', driver = "geojson")

In [337]:
# split = gpd.read_file(r"C://users/kevan/OneDrive/Desktop/Data+/DPS-Planning/GIS_files/pu_with_proj_SPLIT.geojson")
# split = split.sort_values(by="pu_2324_84", ascending=True)

# split = split.reset_index(drop=True)
# split["pu_2324_84"] = split["pu_2324_84"] - 1

# split
# split.to_file(r"C://users/kevan/OneDrive/Desktop/Data+/DPS-Planning-Final/data/pu_split_start_0.geojson")

In [338]:
# read in the planning unit file
pu = gpd.read_file("data/pu_split_start_0.geojson").to_crs(epsg = 4326)
pu = pu[['pu_2324_84', 'basez', 'geometry']]
pu

,pu_2324_84,basez,geometry
0,0,0.0,"POLYGON ((-78.82256 36.19379, -78.82287 36.192..."
1,1,4.0,"POLYGON ((-78.86225 36.05177, -78.85945 36.049..."
2,2,2.0,"POLYGON ((-78.79371 35.94418, -78.79394 35.943..."
3,3,1.0,"POLYGON ((-78.98659 35.88678, -78.98626 35.886..."
4,4,6.0,"POLYGON ((-78.7536 36.03136, -78.74348 36.0254..."
...,...,...,...
846,846,1.0,"POLYGON ((-78.95123 36.10239, -78.95123 36.102..."
847,847,7.0,"POLYGON ((-78.88797 36.04156, -78.88814 36.041..."
848,848,0.0,"POLYGON ((-78.97497 36.05557, -78.97343 36.053..."
849,849,0.0,"POLYGON ((-78.96446 36.09914, -78.96422 36.098..."


In [339]:
joined_2023 = gpd.sjoin(master_devs_2023, pu, how="left", predicate="within")
master_devs_2023["pu_2324_84"] = joined.index_right
joined_2020 = gpd.sjoin(master_devs_2020, pu, how="left", predicate="within")
master_devs_2020["pu_2324_84"] = joined.index_right
master_devs_2023

,A_NUMBER,A_TYPE,A_DATE,A_STATUS,A_STATUS_D,A_PROJECT_,A_DESCRIPT,A_USER_ID,A_CASE_PLA,StatCode,...,duplex_triplex,multifamily,condo,unknown,data_source,geometry,region,stu_gen_100,stu_gen_50,pu_2324_84
0,D2200173,PL_MINSP,2022-05-25,APP,2023-02-01,Pineview Glen Town homes - Mass Grading,Mass grading only site plan for residential to...,JUSTINH,KEAGANSA,APP,...,0.0,0.0,0,0,None,POINT (-78.81051 35.92776),East,0.000000,0.000000,248
1,D2200181,PL_MINSP,2022-06-07,APP,2023-05-16,GTH Owner LLC,33 new town home lots with garages and mail ki...,COLERE,COLERE,APP,...,0.0,0.0,0,0,DRHM,POINT (-78.89255 36.00264),Central,1.088832,0.544416,373
2,D2300150,PL_MINSP,2023-06-07,APP,2024-10-22,Hope Crossing II,32 single family units on small lots and 23 to...,TREYFI,TREYFI,APP,...,0.0,0.0,0,0,DRHM,POINT (-78.84504 35.99883),East,8.593339,4.296670,469
3,D2100111,PL_MINSP,2021-04-12,APP,2024-03-15,Vintage Hill Subdivision,This is a residential subdivision of a 20.96 a...,JESSICADO,COLERE,APP,...,0.0,0.0,0,0,DRHM,POINT (-78.86672 36.10942),North,13.165836,6.582918,595
4,D2300010,PL_MINSP,2023-01-18,APP,2025-08-17,Caring House,Addition of temporary care housing for cancer ...,TREYFI,TREYFI,APP,...,0.0,0.0,0,0,None,POINT (-78.94444 35.97388),Southwest,0.000000,0.000000,390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,D2500199,PL_MINSP,2025-08-11,UN_RE,2025-08-11,Howard's Place,Mixed use multi-family development with access...,HARMONHE,TREYFI,UN_RE,...,0.0,0.0,0,0,None,POINT (-78.76658 35.97569),East,0.000000,0.000000,101
238,D2500202,PL_MINSP,2025-08-14,UN_RE,2025-08-14,443 Infinity Rd,28 unit townhouse community.,ERIKA,TREYFI,UN_RE,...,0.0,0.0,0,0,DRHM,POINT (-78.90088 36.079),North,2.834910,1.417455,196
239,D2500218,PL_MINSP,2025-09-02,UN_RE,2025-09-02,Green Hill,Construction of 440 multi-family units with as...,MICHAELEVE,JALISAHA,UN_RE,...,0.0,440.0,0,0,DRHM,POINT (-78.94701 36.02156),Central,72.444273,36.222136,225
240,D2500219,PL_MINSP,2025-09-04,UN_RE,2025-09-04,Pearson Flats,Multi-family development to be multiple apartm...,JALISAHA,JALISAHA,UN_RE,...,0.0,0.0,0,0,None,POINT (-78.90531 35.95507),Southeast,0.000000,0.000000,289


In [340]:
sums_23 = master_devs_2023.groupby("pu_2324_84", as_index=False)["stu_gen_100"].sum()
sums_23 = sums_23.rename(columns={"stu_gen_100": "master_proj_23"})
pu = pu.merge(sums_23, on="pu_2324_84", how="left")
pu["master_proj_23"] = pu["master_proj_23"].fillna(0)

sums_20 = master_devs_2020.groupby("pu_2324_84", as_index=False)["stu_gen_100"].sum()
sums_20 = sums_20.rename(columns={"stu_gen_100": "master_proj_20"})
pu = pu.merge(sums_20, on="pu_2324_84", how="left")
pu["master_proj_20"] = pu["master_proj_20"].fillna(0)


In [346]:
pu.to_file("data/master_proj.geojson", driver = "GeoJSON")